In [81]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph.message import add_messages
from dotenv import load_dotenv

from langgraph.prebuilt import ToolNode, tools_condition
from ddgs import DDGS
from langchain_core.tools import tool

import os
import requests
import random


In [82]:
load_dotenv()


True

In [83]:
llm = ChatOpenAI(model='gpt-4o-mini')

In [ ]:
# Tools
# Requires: pip install -U ddgs
@tool
def search_tool(query: str) -> str:
    """Search the web using DuckDuckGo.

    Tries multiple DuckDuckGo backends so a single failing backend
    does not break the workflow.
    """
    last_error = None
    for backend in ("google", "bing", "auto", "yahoo"):
        try:
            results = DDGS().text(
                query,
                region="us-en",
                max_results=5,
                backend=backend,
            )
            if results:
                return "\n\n".join(
                    f"{r.get('title', '')}\n{r.get('href', '')}\n{r.get('body', '')}"
                    for r in results
                )
        except Exception as e:
            last_error = e
    return f"Search failed: {last_error}"


@tool
def calculator(first_num: float, second_num: float, operation: str) -> dict:
    """
    Perform a basic arithmetic operation on two numbers.
    Supported operations: add, sub, mul, div
    """
    try:
        if operation == "add":
            result = first_num + second_num
        elif operation == "sub":
            result = first_num - second_num
        elif operation == "mul":
            result = first_num * second_num
        elif operation == "div":
            if second_num == 0:
                return {"error": "Cannot divide by zero"}
            result = first_num / second_num
        else:
            return {"error": f"Unsupported operation: {operation}"}

        return {"result": result}
    except Exception as e:
        return {"error": str(e)}


@tool
def get_stock_price(symbol: str) -> dict:
    """Fetch latest stock price for a given symbol (e.g. 'AAPL', 'TSLA')
    using Alpha Vantage with API key in the URL.
    """
    # Free Alpha Vantage keys allow ~25 requests/day.
    # Get your own key at https://www.alphavantage.co/support/#api-key
    api_key = os.getenv("ALPHAVANTAGE_API_KEY", "KJ7MNUC4W75DNQ62")
    url = (
        "https://www.alphavantage.co/query"
        f"?function=GLOBAL_QUOTE&symbol={symbol}&apikey={api_key}"
    )
    r = requests.get(url)
    return r.json()
    

# Make tool list
tools = [get_stock_price, search_tool, calculator]

# Make the LLM tool-aware
llm_with_tools = llm.bind_tools(tools)


In [89]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


In [90]:
# graph nodes
def chat_node(state: ChatState):
    """LLM node that may answer or request a tool call."""
    messages = state['messages']
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

tool_node = ToolNode(tools, handle_tool_errors=True)  # Executes tool calls


In [91]:
# graph structure
graph = StateGraph(ChatState)
graph.add_node("chat_node", chat_node)
graph.add_node("tools", tool_node)


In [92]:
graph.add_edge(START, "chat_node")

# If the LLM asked for a tool, go to ToolNode; else finish
graph.add_conditional_edges("chat_node", tools_condition)

graph.add_edge("tools", "chat_node")


In [93]:
chatbot = graph.compile()
chatbot.get_graph().draw_mermaid_png()


b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x01y\x00\x00\x01`\x08\x02\x00\x00\x00\xd1\xb8\x9e9\x00\x00\x10\x00IDATx\x9c\xec\xdd\x07xSU\x1b\x07\xf0\x93\x9d\xeeM\'P\xda\xd2\x01e\x17\xd9 C\xa6LA\x04\x17[\x10\x05DA\x96\xb2D@\x11Y\x1f{\x83,e\xc9\x06\x91\x8dl\xcbl\xa1\x94\xd9\xd2\x96Bw\xd3\xec|or\xe8\xf56MKG\x92&\xe1\xfd\xe9\x93\'\xbd+7\xdc\x93\x7f\xce9\xf7\xe6\x1e\xbeF\xa3!\x08!db\\\x82\x10B\xa6\x87Y\x83\x102\x07\xcc\x1a\x84\x909\xf0\tB\x15\x04\xfb\n-\n\x87\xc3!\xa6\x84Y\x83\xcc\x87\t\x97\xc2OP\xc5\x82\xa0\xa1\xc7\x82I\x1c\xa3G\x0ff\r29v\xb2P\xccD\xf6#\xaa\x104S\xf4\x1e\xd9Ac\xac\xd0\xc1\xacA&\xc4D\t[\xe2}I\xfc\xcd\xdc\xcc\xe7jY\x9eZ\xad\xd2(d\xda%\x0b\x14h\x8e\x86h\n\x97o\xd8\x94v"\x97O\xd4J\x03\xd3\x0b(\xb8\x05\rG\xc9\xd1@QW\x17\xee\xa0\xd4h\xd4\x1cN\xa1^K\x8eZ\xbbU\xc3\xbd\x99\x86^N\xa3\xd2M\xe3\x95ha:]\xbb\x87\x06\xb7op\'\x95\x1c\x8e\xe1\xfd/\xbcV\xfe\xc2\xf9\x0c\xffc\xbe\xc2\xe5\xab\xe1\xdfS$\xe6\xb9\xfb\xf0\xab\xd6\xb4\x0f\x08\x11s\ny\xb5\x99r\'\x0e\x07\xbfR\x90)\x14N\x993;\xd3\x1e\

In [94]:
# Regular chat
out = chatbot.invoke({"messages": [HumanMessage(content="Hello!")]})

print(out["messages"][-1].content)


Hello! How can I assist you today?


In [95]:
# Tool call: calculator
out = chatbot.invoke({
    "messages": [HumanMessage(content="What is 25 multiplied by 4?")]
})

for message in out["messages"]:
    print(f"{type(message).__name__}: {message.content}")


HumanMessage: What is 25 multiplied by 4?
AIMessage: 
ToolMessage: {"result": 100.0}
AIMessage: 25 multiplied by 4 is 100.


In [96]:
# Tool call: web search
out = chatbot.invoke({
    "messages": [HumanMessage(content="What is the latest news about LangGraph?")]
})

for message in out["messages"]:
    print(f"{type(message).__name__}: {message.content}")


HumanMessage: What is the latest news about LangGraph?
AIMessage: 
ToolMessage: LangGraph — Latest News, Reports & Analysis | The Hacker News
https://thehackernews.com/search/label/LangGraph
Explore the latest news, real-world incidents, expert analysis, and trends in LangGraph — only on The Hacker News, the leading cybersecurity and IT news platform.

LangGraph: Agent Orchestration Framework for Reliable AI Agents
https://www.langchain.com/langgraph
“LangChain is streets ahead with what they've put forward with LangGraph. LangGraph sets the foundation for how we can build and scale AI workloads — from conversational agents, complex task automation, to custom LLM-backed experiences that 'just work'.

Before You Upgrade to LangGraph in 2026, Read ... | MintSquare
https://www.agentframeworkhub.com/blog/langgraph-news-updates-2026
Everything new in LangGraph v0.2 (2026). Full breakdown of features, breaking changes, code updates, and step-by-step migration tips.

LangGraph Flaw Chain Expo

In [ ]:
# Tool call: stock price
out = chatbot.invoke({
    "messages": [HumanMessage(content="What is the current stock price of AAPL? and ")]
})

for message in out["messages"]:
    print(f"{type(message).__name__}: {message.content}")


HumanMessage: What is the current stock price of AAPL?
AIMessage: 
ToolMessage: {"Global Quote": {"01. symbol": "AAPL", "02. open": "352.9100", "03. high": "355.2800", "04. low": "346.4700", "05. price": "349.2100", "06. volume": "40906860", "07. latest trading day": "2026-11-19", "08. previous close": "352.4500", "09. change": "-3.2400", "10. change percent": "-0.9193%"}}
AIMessage: The current stock price of AAPL (Apple Inc.) is $349.21.


In [98]:
# Inspect the tool calls made by the LLM
for message in out["messages"]:
    if getattr(message, "tool_calls", None):
        print(message.tool_calls)


[{'name': 'get_stock_price', 'args': {'symbol': 'AAPL'}, 'id': 'call_z3Jg1y2GyCIlAR6NDkx7s3v0', 'type': 'tool_call'}]
